# KV cache: đo sự đánh đổi giữa tính lại và lưu K/V

## Mục tiêu và giả thuyết
Cache giữ K/V của prefix để decode không cần tính lại toàn bộ prefix; đổi lại nó dùng thêm bộ nhớ và prefill vẫn cần xử lý toàn bộ prompt. Với mô hình nhỏ chưa huấn luyện, chi phí Python/kernel launch có thể che khuất phần tính toán tiết kiệm, nên cache không được giả định trước là nhanh hơn.

Notebook chạy đúng hai wrapper generation công khai sau khi kiểm tra logits, đồng thời đo controlled replay có continuation IDs giống nhau. Các kết quả là cơ chế suy luận trên trọng số khởi tạo, không đo chất lượng ngôn ngữ hoặc hiệu năng production. Không có bước cài dependency.

Bản notebook thực thi bằng IPython kernel, kèm các hình đã render: [executed notebook](kv_cache_benchmark_runs/20261023T112207Z_coordinator_dca30821/kv_cache_benchmark_executed.ipynb).

In [ ]:
"""Tìm repository root, nạp API và công khai cấu hình thí nghiệm."""
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'llm_foundations' / 'experiments' / 'kv_cache_benchmark.py').exists()), None)
if ROOT is None:
    candidate = Path('../..').resolve()
    if (candidate / 'llm_foundations' / 'experiments' / 'kv_cache_benchmark.py').exists(): ROOT = candidate
if ROOT is None: raise RuntimeError('Run notebook from repository root or experiments directory')
sys.path.insert(0, str(ROOT))
from llm_foundations.experiments.kv_cache_benchmark import (
    BenchmarkConfig,
    plot_results,
    run_benchmark,
    save_results,
)

config = BenchmarkConfig()
print('Repository:', ROOT)
print('Config:', json.dumps(config.__dict__, default=lambda x: x.__dict__, indent=2))
print('CUDA available:', __import__('torch').cuda.is_available())

## Ranh giới đo

Mỗi workload so sánh cùng prompt, token budget, model weights và dtype. Correctness gate chạy trước thời gian; phase timing ở `eval()`/`no_grad()`, có CUDA synchronization ở biên interval. Decode cache được tạo trước khi bắt đầu đo D bước. Wrapper timing vẫn gọi nguyên wrapper để giữ nguyên xử lý mode, selector và nối token. Bộ nhớ CUDA chạy riêng từng arm, chỉ để một mô hình trên GPU, và peak allocated không cộng thêm KV payload. Quantile p25/p75 dùng nội suy tuyến tính tại `(n-1)*p`.

## Cách đọc các metric

Với prompt dài P token, budget B token (số token mới yêu cầu sinh tối đa) và capacity M: `N=min(B, M-P)` là số token thực sinh, `D=max(N-1, 0)` là số decode sau prefill, và cache cuối có chiều dài `T=P+D`.

| Metric | Câu hỏi | Công thức / đơn vị | Đo gì, bỏ gì | Cách đọc |
|---|---|---|---|---|
| Logit max error / allclose | Hai đường tính có cho logits tương đương? | `max(abs(logits_ref-logits_cache))`; allclose dùng `atol`, `rtol` | So logits prefill và từng token decode; không phải timing | Error nhỏ và allclose=True cho phép diễn giải timing; đây không phải chất lượng ngôn ngữ |
| Prefill | Bao lâu để xử lý prompt? | Thời gian mỗi prompt call, ms | Uncached: full prompt forward; cached: logits và K/V cache; selector bị loại | Cache không bỏ chi phí đọc toàn prompt |
| Controlled decode | Cho cùng lịch sử token, hai cách tính bước tiếp theo mất bao lâu? | `decode_total_ms / D`, ms/step | Đo D bước replay trên cùng continuation IDs; chuẩn bị prompt/cache ban đầu ở ngoài interval | Là trung bình qua các context length; N=1 thì D=0 và metric là N/A, không phải 0 |
| Real-wrapper generation | Người gọi thực tế chờ bao lâu? | Wall time mỗi wrapper, ms | Gọi wrapper công khai; gồm prefill/forward, selector, và append ID, bỏ prompt clone ngoài interval | Đây là end-to-end generation của workload, không cộng với phase timings |
| Tokens/s | Tốc độ sinh quan sát được? | `N / (generation_ms / 1000)`, tokens/s | Dùng chính wall time wrapper và N thực sinh | N=0 thì không xác định; không dùng budget thay N khi bị capacity truncate |
| Speedup | Arm nào nhanh hơn cùng workload? | `median(uncached) / median(cached)`, ratio-of-medians | Chia median cùng metric/case; không chia các quartile | >1 nghiêng về cache, <1 nghiêng về uncached; không hàm ý mọi prompt đều vậy |
| K/V MiB | Cache payload logic lớn bao nhiêu? | `sum(numel * element_size) / 2^20`, MiB tại T=P+D | Chỉ tensor K/V; không gồm allocator/workspace/model | Mức tăng theo context là payload, không phải toàn bộ VRAM |
| CUDA peak / delta | Bao nhiêu tensor memory được allocator giữ? | `peak_allocated_mib`; delta=`peak-baseline`, MiB | Pass riêng từng arm; peak trong wrapper generation, baseline gồm mô hình+prompt và pre-model allocated footprint | Không cộng K/V MiB vào peak. Footprint pre-model đo được là 8.125 MiB; được ghi trong baseline, không quy nó cho K/V |

Note: 
| Metric | Đang đếm gì? |
|---|---|
| K/V MiB | Chỉ dữ liệu của các tensor K và V đang giữ |
| Peak allocated MiB | Đỉnh tổng bộ nhớ tensor do allocator PyTorch theo dõi trong lần chạy |
| Peak delta MiB | Peak trừ baseline đã có trước khi gọi generation |

## Quy trình đo và giới hạn thống kê

1. Sao chép trọng số theo role, đối chiếu parameter mapping, rồi chặn benchmark nếu logits không allclose.
2. Dùng cùng IDs/history cho controlled replay; warm up từng arm/workload và tạo cache mới cho mỗi repetition.
3. Đồng bộ CUDA ngay trước/sau interval; không đồng bộ từng token trong phase loop. Đo controlled prefill/decode riêng khỏi wrapper generation.
4. Lặp lại, luân phiên arm order và báo median cùng p25-p75; đây là độ phân tán quan sát trong các lần lặp, **không phải confidence interval**.
5. Đo memory ở pass tách biệt, chỉ một arm trên GPU; ghi pre-model, baseline, peak và delta để thấy phần footprint đã tồn tại trước model.

`config.seed` tạo input token IDs; seed khởi tạo model là metadata riêng `weight_seed` (cố định 20261023), không phải `config.seed`. Trọng số hai kiến trúc được copy/match trực tiếp, không dựa vào seed giống nhau.

Đồng thời, run này cố định greedy cho cả hai bên wrapper, được ghi trực tiếp trong _call_arm.

In [ ]:
"""Chạy sweep bounded mặc định, lưu JSON/CSV và SVG vào run directory mới."""
import torch
from IPython.display import Markdown, display

result = run_benchmark(config)
run_dir = save_results(result, ROOT / 'llm_foundations/experiments/kv_cache_benchmark_runs')
figures = plot_results(result, output_dir=run_dir / 'graphs')
print('Run:', run_dir)
print('Device:', result['environment']['device'], '| GPU:', result['environment']['gpu'])
print('Warmup/repeats:', config.warmup, '/', config.repeats)
print('Parameter mapping count:', result['mapping']['count'], '| correctness:', all(x.get('allclose') is not False for x in result['correctness']))
print('Source revision and hashes:', json.dumps(result['environment']['source'], indent=2))
print('Observed:', result['observations_vi'])

In [ ]:
"""Hiển thị các figure từ samples đã lưu và tóm tắt đúng từng workload."""
for name, figure in figures.items():
    print(name)
    display(figure)
for case in result['cases']:
    print(case['case_id'], 'sweep=', case['sweep'], 'P=', case['prompt_length'], 'B=', case['requested_tokens'], 'N=', case['actual_tokens'], 'decode steps=', case['decode_steps'], 'IDs agree=', case['generation_ids_agree'])
    print('  speedups (ratio of medians):', case['speedups'])
    print('  cache payload MiB:', case['cache']['kv_mib'], '| cached peak/delta MiB:', case['memory']['cached']['peak_allocated_mib'], case['memory']['cached']['peak_delta_mib'])
display(Markdown('### Diễn giải: ' + result['observations_vi'] + ' `kv_mib` là kích thước tensor K/V logic tại T=P+D, không phải tổng memory. `peak_allocated_mib` và `peak_delta_mib` đo live tensor allocation của allocator trong lượt generation cô lập; chúng không được cộng với payload. B=1 không có decode step nên decode là N/A, không phải 0. ID agreement chỉ mô tả greedy output của hai model đã matched trong workload này.'))